# Migrate survey rows from CSV

This notebook updates existing `surveys` rows using the composite key `survey_id` + `respondent_id`.

- The default is a dry run. Set `DRY_RUN = False` only after reviewing the summary.
- CSV rows with no matching database row are skipped and reported in `unmatched_df`.
- Duplicate database matches are skipped and reported in `ambiguous_df`.
- `INSERT_UNMATCHED` is available as an explicit opt-in, but stays disabled by default.
- The full source CSV row is saved in `raw_row_data`; direct survey fields are updated from the mapped source columns.
- Rows are processed in parallel chunks with one database session per worker; progress, commits, failures, elapsed time, and throughput are logged.


In [ ]:
from pathlib import Path
from datetime import date, datetime
import logging
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd

# Change this to the CSV file being migrated.
CSV_PATH = Path(
    "/Users/jimmytse/Documents/CLSense-Backend/notebooks/docs/ext_cls_data_202607_wtcmy_20260819_131354.csv"
)
SURVEY_ORDER_CUTOFF = datetime(2026, 7, 31, 23, 59, 59, tzinfo=datetime.timezone.utc)  # UTC+0
FILTERED_CSV_PATH = None  # Defaults beside CSV_PATH.
DRY_RUN = False
INSERT_UNMATCHED = False
BATCH_SIZE = 500
MAX_WORKERS = max(1, int(os.getenv("MIGRATION_MAX_WORKERS", "8")))

# The notebook can be opened from the repository root or from the notebooks directory.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "backend").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
BACKEND_ROOT = PROJECT_ROOT / "backend"
sys.path.insert(0, str(BACKEND_ROOT))

from models.Survey import Survey
from utils.database import SessionLocal

logger = logging.getLogger("survey_csv_migration")
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(
        logging.Formatter("%(asctime)s %(levelname)s [%(threadName)s] %(message)s")
    )
    logger.addHandler(handler)
logger.setLevel(logging.INFO)
logger.propagate = False
if FILTERED_CSV_PATH is None:
    FILTERED_CSV_PATH = CSV_PATH.with_name(
        f"{CSV_PATH.stem}-survey-order-after-{SURVEY_ORDER_CUTOFF.isoformat()}.csv"
    )
else:
    FILTERED_CSV_PATH = Path(FILTERED_CSV_PATH)

if not CSV_PATH.is_file():
    raise FileNotFoundError(f"CSV file does not exist: {CSV_PATH}")

print(f"Project root: {PROJECT_ROOT}")
print(f"CSV: {CSV_PATH}")
print(f"Dry run: {DRY_RUN}")
print(f"Insert unmatched: {INSERT_UNMATCHED}")
print(f"Survey order date cutoff: {SURVEY_ORDER_CUTOFF.isoformat()}")
print(f"Filtered CSV output: {FILTERED_CSV_PATH}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Worker threads: {MAX_WORKERS}")

Project root: /Users/jimmytse/Documents/CLSense-Backend
CSV: /Users/jimmytse/Documents/CLSense-Backend/notebooks/docs/ext_cls_data_202607_wtcmy_20260819_131354.csv
Dry run: True
Insert unmatched: False
Batch size: 500
Worker threads: 8


In [2]:
def normalize_key(value):
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None
    # Remove only a pandas-style trailing .0; preserve meaningful leading zeroes.
    if text.endswith(".0") and text[:-2].isdigit():
        return text[:-2]
    return text


def nullable_float(value):
    if value is None or pd.isna(value) or str(value).strip() == "":
        return None
    return float(value)


def nullable_int(value):
    parsed = nullable_float(value)
    return None if parsed is None else int(parsed)


def nullable_datetime(value):
    if value is None or pd.isna(value) or str(value).strip() == "":
        return None
    parsed = pd.to_datetime(value, utc=True, errors="coerce")
    if pd.isna(parsed):
        raise ValueError(f"Invalid datetime: {value!r}")
    return parsed.to_pydatetime()


def nullable_bool_from_delete_flag(value):
    if value is None or pd.isna(value) or str(value).strip() == "":
        return False
    return str(value).strip().upper() == "Y"


def source_value(row, *column_names):
    for column_name in column_names:
        if column_name in row.index:
            value = row[column_name]
            if isinstance(value, pd.Series):
                return value.iloc[0]
            return value
    return None

In [3]:
# Read IDs as strings so leading zeroes and large identifiers are not changed.
df = pd.read_csv(
    CSV_PATH,
    dtype=str,
    keep_default_na=False,
    na_values=[""],
    encoding="utf-8",
)

required_columns = {"survey_id", "respondent_id"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"CSV is missing required columns: {sorted(missing_columns)}")

if "survey_order_date" not in df.columns:
    raise ValueError("CSV is missing required column: survey_order_date")

source_survey_order_dates = df["survey_order_date"]
if isinstance(source_survey_order_dates, pd.DataFrame):
    logger.warning("CSV contains duplicate survey_order_date columns; using the first one for filtering")
    source_survey_order_dates = source_survey_order_dates.iloc[:, 0]
parsed_survey_order_dates = pd.to_datetime(source_survey_order_dates, utc=True, errors="coerce")
recent_row_mask = parsed_survey_order_dates.dt.date > SURVEY_ORDER_CUTOFF
recent_rows_df = df.loc[recent_row_mask].copy()
FILTERED_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
recent_rows_df.to_csv(FILTERED_CSV_PATH, index=False, encoding="utf-8")
logger.info("Saved %s rows with survey_order_date after %s to %s", len(recent_rows_df), SURVEY_ORDER_CUTOFF.isoformat(), FILTERED_CSV_PATH)

df["_survey_key"] = df["survey_id"].map(normalize_key)
df["_respondent_key"] = df["respondent_id"].map(normalize_key)
invalid_key_rows = df[df["_survey_key"].isna() | df["_respondent_key"].isna()].copy()
work_df = df[df["_survey_key"].notna() & df["_respondent_key"].notna()].copy()

print(f"Read {len(df):,} CSV rows")
print(f"Rows with usable composite keys: {len(work_df):,}")
print(f"Rows with missing composite keys: {len(invalid_key_rows):,}")
print(f"Rows saved to filtered CSV: {len(recent_rows_df):,}")
display(recent_rows_df.head())
display(df.head())

Read 94,539 CSV rows
Rows with usable composite keys: 94,539
Rows with missing composite keys: 0


,survey_type,survey_id,respondent_id,bu_key,store_key,question_id,submitdate,survey_order_date,reporting_month,etl_last_update,answer,CLS,is_delete,_survey_key,_respondent_key
0,cls_long_survey,646344,1295645,WTCMY,2312,Q6(comment),2026-08-16T23:51:28.000Z,2026-08-15T16:21:15.000Z,202608,2026-08-16T19:13:08.646Z,be able to get local n korea branding in one s...,100.0,N,646344,1295645
1,cls_long_survey,646344,1295644,WTCMY,256,Q6(comment),2026-08-16T23:37:36.000Z,2026-08-15T11:54:49.000Z,202608,2026-08-16T19:13:08.646Z,Wide range of products,100.0,N,646344,1295644
2,cls_long_survey,646344,1295640,WTCMY,666,Q6(comment),2026-08-16T23:14:57.000Z,2026-08-12T18:32:20.000Z,202608,2026-08-16T19:13:08.646Z,Keberadaan kakitangan yang ada tengah menyusun...,100.0,N,646344,1295640
3,cls_long_survey,646344,1295639,WTCMY,349,Q6(comment),2026-08-16T23:09:15.000Z,2026-08-14T20:59:21.000Z,202608,2026-08-16T19:13:08.646Z,The price shown on shelf (promo price) and in ...,92.8,N,646344,1295639
4,cls_long_survey,646344,1295637,WTCMY,619,Q6(comment),2026-08-16T23:08:24.000Z,2026-08-14T12:20:12.000Z,202608,2026-08-16T19:13:08.646Z,More promotional items on display.,93.0,N,646344,1295637


## Field mapping

The CSV contains more fields than the `surveys` table. The mapped fields below are updated directly. `survey_type`, `question_id`, `reporting_month`, `bu_key`, and other source-only fields remain available inside `raw_row_data`. Derived sentiment, topic, keyword, and department relationships are intentionally left unchanged.

In [4]:
def build_update_values(row):
    # CLS is the canonical score field; accept CSL for legacy extracts.
    cls_raw = source_value(row, "CLS", "CSL", "cls", "csl")
    values = {
        "store_key": nullable_int(source_value(row, "store_key")),
        "comment": source_value(row, "answer", "comment"),
        "reported_at": nullable_datetime(source_value(row, "survey_order_date", "reported_at")),
        "updated_at": nullable_datetime(source_value(row, "etl_last_update", "updated_at")),
        "is_deleted": nullable_bool_from_delete_flag(source_value(row, "is_delete", "is_deleted")),
        "cls": nullable_float(cls_raw),
        "raw_row_data": {
            "migration_source": str(CSV_PATH),
            "row": {str(key): (None if pd.isna(value) else str(value)) for key, value in row.items() if not str(key).startswith("_")},
        },
    }
    # Do not overwrite a field from a source column that is not present.
    if "store_key" not in row.index:
        values.pop("store_key")
    if not ({"answer", "comment"} & set(row.index)):
        values.pop("comment")
    if not ({"survey_order_date", "reported_at"} & set(row.index)):
        values.pop("reported_at")
    if not ({"etl_last_update", "updated_at"} & set(row.index)):
        values.pop("updated_at")
    if not ({"is_delete", "is_deleted"} & set(row.index)):
        values.pop("is_deleted")
    if not ({"CLS", "CSL", "cls", "csl"} & set(row.index)):
        values.pop("cls")
    return values

In [5]:
def chunked(items, chunk_size):
    for start in range(0, len(items), chunk_size):
        yield items[start:start + chunk_size]


def process_chunk(chunk_number, chunk):
    # A SQLAlchemy Session is never shared between worker threads.
    db = SessionLocal()
    results = []
    update_mappings = []
    insert_objects = []
    try:
        logger.info("Worker started chunk %s with %s rows", chunk_number, len(chunk))
        for row_number, row in chunk:
            survey_id = row["_survey_key"]
            respondent_id = row["_respondent_key"]
            base_result = {"csv_row": row_number, "survey_id": survey_id, "respondent_id": respondent_id}
            try:
                matches = (
                    db.query(Survey.id)
                    .filter(
                        Survey.survey_id == survey_id,
                        Survey.respondent_id == respondent_id,
                    )
                    .all()
                )

                if len(matches) > 1:
                    results.append({**base_result, "status": "ambiguous", "matches": len(matches)})
                    logger.warning("Chunk %s row %s skipped: %s database matches for key (%s, %s)", chunk_number, row_number, len(matches), survey_id, respondent_id)
                    continue

                values = build_update_values(row)
                if not matches:
                    result = {**base_result, "status": "unmatched"}
                    if not INSERT_UNMATCHED:
                        results.append(result)
                        continue
                    if values.get("store_key") is None:
                        results.append({**base_result, "status": "failed", "error": "store_key is required for insert"})
                        continue
                    insert_objects.append(Survey(survey_id=survey_id, respondent_id=respondent_id, **values))
                    result["status"] = "inserted"
                    results.append(result)
                else:
                    database_id = matches[0][0]
                    update_mappings.append({"id": database_id, **values})
                    results.append({**base_result, "status": "updated", "database_id": database_id})
            except Exception as error:
                db.rollback()
                logger.exception("Chunk %s row %s failed for key (%s, %s)", chunk_number, row_number, survey_id, respondent_id)
                results.append({**base_result, "status": "failed", "error": str(error)})

        write_count = len(update_mappings) + len(insert_objects)
        if DRY_RUN:
            db.rollback()
            logger.info("Worker completed chunk %s in dry-run mode: %s candidate writes", chunk_number, write_count)
        else:
            if update_mappings:
                db.bulk_update_mappings(Survey, update_mappings)
            if insert_objects:
                db.add_all(insert_objects)
            if write_count:
                db.commit()
            logger.info("Worker committed chunk %s: %s updates/inserts", chunk_number, write_count)
    except Exception as error:
        db.rollback()
        logger.exception("Chunk %s rolled back: %s", chunk_number, error)
        for result in results:
            if result["status"] in {"updated", "inserted"}:
                result["status"] = "failed"
                result["error"] = f"Chunk rolled back: {error}"
    finally:
        db.close()
    return results


started_at = time.monotonic()
work_items = [(row_number, row) for row_number, (_, row) in enumerate(work_df.iterrows(), start=2)]
chunks = list(chunked(work_items, BATCH_SIZE))
worker_count = min(MAX_WORKERS, max(1, len(chunks)))
all_results = []
logger.info("Starting migration: %s rows, %s chunks, %s worker threads, dry_run=%s", len(work_items), len(chunks), worker_count, DRY_RUN)

with ThreadPoolExecutor(max_workers=worker_count, thread_name_prefix="survey-migration") as executor:
    futures = [executor.submit(process_chunk, chunk_number, chunk) for chunk_number, chunk in enumerate(chunks, start=1)]
    for completed_chunks, future in enumerate(as_completed(futures), start=1):
        all_results.extend(future.result())
        processed = min(completed_chunks * BATCH_SIZE, len(work_items))
        elapsed = time.monotonic() - started_at
        rate = processed / elapsed if elapsed else 0
        logger.info("Progress: %s/%s rows (%.1f%%), %.1f rows/sec", processed, len(work_items), (processed / len(work_items) * 100) if work_items else 100, rate)

updated_rows = [result for result in all_results if result["status"] == "updated"]
unmatched_rows = [result for result in all_results if result["status"] == "unmatched"]
ambiguous_rows = [result for result in all_results if result["status"] == "ambiguous"]
inserted_rows = [result for result in all_results if result["status"] == "inserted"]
failed_rows = [result for result in all_results if result["status"] == "failed"]
elapsed = time.monotonic() - started_at
logger.info("Migration finished in %.2f seconds: updated=%s, unmatched=%s, ambiguous=%s, failed=%s", elapsed, len(updated_rows), len(unmatched_rows), len(ambiguous_rows), len(failed_rows))
print(f"Matched rows updated: {len(updated_rows):,}")
print(f"Unmatched rows: {len(unmatched_rows):,}")
print(f"Ambiguous rows skipped: {len(ambiguous_rows):,}")
print(f"Rows inserted: {len(inserted_rows):,}")
print(f"Rows failed: {len(failed_rows):,}")

2026-08-19 16:24:50,728 INFO [MainThread] Starting migration: 94539 rows, 190 chunks, 8 worker threads, dry_run=True
2026-08-19 16:24:50,728 INFO [survey-migration_0] Worker started chunk 1 with 500 rows
2026-08-19 16:24:50,729 INFO [survey-migration_1] Worker started chunk 2 with 500 rows
2026-08-19 16:24:50,729 INFO [survey-migration_2] Worker started chunk 3 with 500 rows
2026-08-19 16:24:50,729 INFO [survey-migration_3] Worker started chunk 4 with 500 rows
2026-08-19 16:24:50,731 INFO [survey-migration_4] Worker started chunk 5 with 500 rows
2026-08-19 16:24:50,731 INFO [survey-migration_5] Worker started chunk 6 with 500 rows
2026-08-19 16:24:50,731 INFO [survey-migration_6] Worker started chunk 7 with 500 rows
2026-08-19 16:24:50,731 INFO [survey-migration_7] Worker started chunk 8 with 500 rows
2026-08-19 16:24:56,898 INFO [survey-migration_5] Worker completed chunk 6 in dry-run mode: 413 candidate writes
2026-08-19 16:24:56,901 INFO [survey-migration_5] Worker started chunk 9 w

Matched rows updated: 75,080
Unmatched rows: 19,459
Ambiguous rows skipped: 0
Rows inserted: 0
Rows failed: 0


In [6]:
unmatched_df = pd.DataFrame(unmatched_rows)
ambiguous_df = pd.DataFrame(ambiguous_rows)
failed_df = pd.DataFrame(failed_rows)
updated_df = pd.DataFrame(updated_rows)
inserted_df = pd.DataFrame(inserted_rows)

# Review these before changing DRY_RUN to False.
display(unmatched_df.head(20))
display(ambiguous_df.head(20))
display(failed_df.head(20))

,csv_row,survey_id,respondent_id,status
0,2515,646344,1288967,unmatched
1,2523,646344,1288946,unmatched
2,2527,646344,1288937,unmatched
3,2533,646344,1288921,unmatched
4,2535,646344,1288916,unmatched
5,2536,646344,1288914,unmatched
6,2538,646344,1288913,unmatched
7,2540,646344,1288910,unmatched
8,2541,646344,1288908,unmatched
9,2544,646344,1288894,unmatched


""


""


## Commit checklist

1. Confirm the CSV path and database environment.
2. Run with `DRY_RUN = True` and review `updated_df`, `unmatched_df`, `ambiguous_df`, and `failed_df`.
3. If the unmatched rows should be created, set `INSERT_UNMATCHED = True` and confirm each row has a valid existing `store_key`.
4. Set `DRY_RUN = False` to commit the update in batches.